# The Dreamlands: train the Lovecraft style adapter

1. **Runtime → Change runtime type → T4 GPU** (free tier is fine).
2. Run the cells in order. Training takes very roughly an hour on a T4.
3. The adapter is saved to your Google Drive so a disconnect doesn't lose it. Re-run the training cell with `--resume` to continue.

The repo must contain `tools/story/` (push it to GitHub first). If the repo is private, use the upload option in cell 2 instead.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Option A: public repo
!git clone --depth 1 https://github.com/Soul-8691/Dream-Quest.git /content/repo
# Option B: private repo -> zip the project locally (with the PDF and tools/story), upload it, then:
# from google.colab import files; files.upload()   # pick the zip
# !unzip -q *.zip -d /content/repo
%cd /content/repo

In [ ]:
!pip install -q -r tools/story/requirements-ml.txt

In [ ]:
# Build the filtered dataset from the PDF (takes a minute or two)
!python tools/story/extract_corpus.py
!python tools/story/build_dataset.py
!python tools/story/style_profile.py > /dev/null

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ADAPTER = '/content/drive/MyDrive/dreamlands-lora'

In [ ]:
# Smoke test first (2-3 minutes). If it runs without errors, run the full training below.
!python tools/story/train_lora.py --max-steps 5 --out-dir /content/smoke-test

In [ ]:
# Full training. After a disconnect, re-run cells 2-5 and add --resume here.
!python tools/story/train_lora.py --out-dir "$ADAPTER"

In [ ]:
# Try it: compare the plain base model with the adapter
SEED = 'Then the sealer lifted the sign against the opened sky, and'
!python tools/story/generate.py --prompt "$SEED" --n 3 --adapter "$ADAPTER" --no-score
!cat tools/story/out/candidates/prompt.md

In [ ]:
# Draft every beat marked in STORY.md, scored and ranked. Download the results.
!python tools/story/generate.py --all --n 6 --adapter "$ADAPTER"
!cd tools/story/out && zip -qr /content/candidates.zip candidates
from google.colab import files; files.download('/content/candidates.zip')